# Fase 3: Deteksi Anomali (Penjaga Garis Depan)

## Tujuan
Membuat alarm dini menggunakan Unsupervised Learning (Isolation Forest) untuk mendeteksi perilaku sensor yang tidak wajar.

## Langkah-langkah
- 3.1: Normalisasi Data (StandardScaler / MinMaxScaler)
- 3.2: Training Isolation Forest
- 3.3: Penerapan Aturan Bisnis Rolling Window (3 jam berturut-turut)
- 3.4: Evaluasi & Visualisasi Deteksi Anomali
- 3.5: Ekspor Model (anomaly_detector.pkl)

In [ ]:
import pandas as pd

print("Melakukan Sanity Check pada Data Ekspor (Fase 3)...\n")

# Memuat data hasil Fase 2 dengan menjadikan timestamp sebagai index bertipe datetime
df_ready = pd.read_csv('../data/processed/sensor_features_engineered.csv', index_col='timestamp', parse_dates=True)

# Validasi Dimensi Data
print(f"Bentuk Dataset (Shape): {df_ready.shape}")

# Validasi Kekosongan Data (Tidak boleh ada NaN)
max_nan = df_ready.isnull().sum().max()
print(f"Jumlah nilai NaN terbanyak pada salah satu kolom: {max_nan}")

# Menampilkan 5 baris pertama
print("\nSekilas isi df_ready:")
display(df_ready.head())

In [ ]:
# --- 3.1 Persiapan Data & Normalisasi ---

# 1. Buang sisa baris yang memiliki nilai kosong (NaN) pasca-rolling/FFT
df_ready.dropna(inplace=True)
print(f"Dimensi data setelah dropna: {df_ready.shape}")

# 2. Import StandardScaler
from sklearn.preprocessing import StandardScaler

# 3. Pisahkan label jawaban (Simpan kolom 'failure' ke variabel y)
# Kita pastikan kita tidak terkena error jika sewaktu-waktu kolomnya berganti nama.
if 'failure' in df_ready.columns:
    y = df_ready['failure']
else:
    y = None
    print("Himbauan: Kolom 'failure' tidak ditemukan. Mode Unsupervised murni.")

# 4. Buat variabel fitur X dengan membuang kolom target dan kolom teks ('machine_id')
cols_to_drop = ['machine_id']
if 'failure' in df_ready.columns:
    cols_to_drop.append('failure')
    
X = df_ready.drop(columns=[col for col in cols_to_drop if col in df_ready.columns])

# 5. Inisialisasi StandardScaler dan Lakukan Normalisasi pada X
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Mengubah array X_scaled menjadi DataFrame untuk visualisasi hasil
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("\n--- Validasi Tabel Hasil Normalisasi (StandardScaler) ---")
display(X_scaled_df.head(5))